In [ ]:
# process to parquet 
import os
import sys
from pathlib import Path
import re
import zipfile
import zarr
import shutil
import torch
from zarr.storage import ZipStore
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from omegaconf import OmegaConf
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print(os.cpu_count())
# -------------------------
# Add external repos
# -------------------------
BASE = Path("/projappl/project_2012747/mars_derrick_branch/third_party")
CKPT_DIR = Path("/scratch/project_2012747/Mars_Derrick/checkpoints/checkpoint_downsample_100")
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/intermediate/")
OUT_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

sys.path.insert(0, str(BASE / "latent_diffusion"))
sys.path.insert(0, str(BASE / "taming-transformers"))

OUT_DIR.mkdir(exist_ok=True)

# -------------------------
# Lightning wrapper
# -------------------------
class VQForOrders(pl.LightningModule):
    def __init__(self, vqmodel):
        super().__init__()
        self.m = vqmodel

    def forward(self, x):
        q, _, _ = self.m.encode(x)
        return self.m.decode(q)

# -------------------------
# Load best checkpoint
# -------------------------
def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        return float(m.group(1))
    return min(ckpts, key=extract_val_loss)

def load_model(best_ckpt):
    from ldm.util import instantiate_from_config
    cfg = OmegaConf.load(BASE / "latent_diffusion/models/first_stage_models/vq-f4/config.yaml")
    vq = instantiate_from_config(cfg.model)
    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt,
        vqmodel=vq,
        strict=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, device

# -------------------------
# Fast Zarr dataset
# -------------------------
class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)

# -------------------------
# Processing function
# -------------------------
def process_zip_file(zip_path: Path, model, device, batch_size=512):
    print(f"Processing {zip_path.name} ...")

    # extract to temp folder
    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    # the extracted folder should be a .zarr folder
    zarr_folders = [p for p in extract_dir.iterdir() if p.is_dir() and p.suffix == ".zarr"]
    if not zarr_folders:
        # fallback: maybe the folder itself is the zarr
        zarr_folder = extract_dir
    else:
        zarr_folder = zarr_folders[0]

    print(f"Using extracted folder: {zarr_folder}")

    # open Zarr
    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]
    num_samples = arr.shape[0]

    all_tokens = []

    dataset = ZarrDataset(arr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=os.cpu_count(), pin_memory=True)

    with torch.inference_mode():
        for i, x in enumerate(loader):
            x = x.to(device, non_blocking=True)
            with torch.autocast("cuda"):
                _, _, info = model.m.encode(x)
            B = x.size(0)
            tokens = info[2].view(B, -1).cpu().numpy()
            all_tokens.append(tokens)

            if i % 20 == 0:
                print(f"{zip_path.name}: processed {min((i+1)*batch_size, num_samples)}/{num_samples}")

    all_tokens = np.vstack(all_tokens)

    # save as .zarr folder
    out_zarr_folder = OUT_DIR / f"{zip_path.stem.replace('_order_images','')}_64_vector.zarr"
    if out_zarr_folder.exists():
        shutil.rmtree(out_zarr_folder)
    out_root = zarr.open(str(out_zarr_folder), mode="w", shape=all_tokens.shape,
                         chunks=(batch_size, all_tokens.shape[1]), dtype=all_tokens.dtype)
    out_root[:] = all_tokens

    # zip the output
    shutil.make_archive(str(out_zarr_folder), 'zip', root_dir=out_zarr_folder)
    print(f"Saved zipped Zarr: {str(out_zarr_folder)}.zip")

    # cleanup
    shutil.rmtree(extract_dir)

# -------------------------
# Main
# -------------------------
def main():
    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))
    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

if __name__ == "__main__":
    main()


In [5]:
# process to parquet 
import os
import sys
from pathlib import Path
import re
import zipfile
import zarr
import shutil
import torch
from zarr.storage import ZipStore
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from omegaconf import OmegaConf
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

print(os.cpu_count())
# -------------------------
# Add external repos
# -------------------------
BASE = Path("/projappl/project_2012747/mars_derrick_branch/third_party")
CKPT_DIR = Path("/scratch/project_2012747/Mars_Derrick/checkpoints/checkpoint_downsample_100")
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/intermediate/")
OUT_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

sys.path.insert(0, str(BASE / "latent_diffusion"))
sys.path.insert(0, str(BASE / "taming-transformers"))

OUT_DIR.mkdir(exist_ok=True)

# -------------------------
# Lightning wrapper
# -------------------------
class VQForOrders(pl.LightningModule):
    def __init__(self, vqmodel):
        super().__init__()
        self.m = vqmodel

    def forward(self, x):
        q, _, _ = self.m.encode(x)
        return self.m.decode(q)

# -------------------------
# Load best checkpoint
# -------------------------
def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        return float(m.group(1))
    return min(ckpts, key=extract_val_loss)

def load_model(best_ckpt):
    from ldm.util import instantiate_from_config
    cfg = OmegaConf.load(BASE / "latent_diffusion/models/first_stage_models/vq-f4/config.yaml")
    vq = instantiate_from_config(cfg.model)
    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt,
        vqmodel=vq,
        strict=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, device

import os
import shutil
import zipfile
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import zarr
import numpy as np
from tqdm import tqdm

class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)



def process_zip_file(zip_path: Path, model, device, batch_size=1024, num_saves=10):
    print(f"Processing {zip_path.name} ...")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    zarr_folders = [p for p in extract_dir.iterdir() if p.is_dir() and p.suffix == ".zarr"]
    zarr_folder = zarr_folders[0] if zarr_folders else extract_dir
    print(f"Using extracted folder: {zarr_folder}")

    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]
    num_samples = arr.shape[0]

    with torch.inference_mode():
        sample_input = torch.zeros((1, *arr.shape[1:]), device=device)
        _, _, info = model.m.encode(sample_input)
        sample_tokens = info[2]
        if isinstance(sample_tokens, (tuple, list)):
            sample_tokens = sample_tokens[0]
        sample_shape = sample_tokens.numel()

    out_zarr_folder = OUT_DIR / f"{zip_path.stem.replace('_order_images','')}_64_vector.zarr"
    if out_zarr_folder.exists():
        shutil.rmtree(out_zarr_folder)

    out_root = zarr.open(
        str(out_zarr_folder),
        mode="w",
        shape=(num_samples, sample_shape),
        chunks=(batch_size, sample_shape),
        dtype=np.float32,
        compressor=zarr.LZ4()
    )

    dataset = ZarrDataset(arr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        num_workers=min(30, os.cpu_count()), pin_memory=True)

    batches_per_save = math.ceil(len(loader) / num_saves)
    batch_idx = 0
    start_idx = 0

    with torch.inference_mode():
        for x in tqdm(loader, desc=f"Processing {zip_path.name}"):
            x = x.to(device, non_blocking=True)
            with torch.autocast("cuda"):
                _, _, info = model.m.encode(x)
            batch_tokens = info[2]
            if isinstance(batch_tokens, (tuple, list)):
                batch_tokens = batch_tokens[0]
            batch_tokens = batch_tokens.view(x.size(0), -1).cpu().numpy()
            out_root[start_idx:start_idx+batch_tokens.shape[0], :] = batch_tokens
            start_idx += batch_tokens.shape[0]
            batch_idx += 1

            if batch_idx % batches_per_save == 0:
                print(f"Saved progress at batch {batch_idx}/{len(loader)}")

    shutil.make_archive(str(out_zarr_folder), 'zip', root_dir=out_zarr_folder)
    print(f"Zipped Zarr: {str(out_zarr_folder)}.zip")
    shutil.rmtree(extract_dir)
    
# -------------------------
# Main
# -------------------------
def main():
    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))
    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

if __name__ == "__main__":
    main()


40
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Found 270 zip files to process.
Processing GOOG_2025-11-04_order_images.zarr.zip ...


/users/edwardma/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 30 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Using extracted folder: /scratch/project_2012747/mars_data/order_batch_model/train/intermediate/GOOG_2025-11-04_order_images.zarr


Processing GOOG_2025-11-04_order_images.zarr.zip:   0%|          | 0/6828 [00:00<?, ?it/s]/users/edwardma/.local/lib/python3.12/site-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 30 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Processing GOOG_2025-11-04_order_images.zarr.zip:   7%|▋         | 510/6828 [05:44<1:11:02,  1.48it/s]


KeyboardInterrupt: 

In [7]:
import os
import math
import shutil
import zipfile
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import zarr
import numpy as np
from tqdm import tqdm

class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)


def process_zip_file(zip_path: Path, model, device, batch_size=1024, num_saves=10):
    print(f"Processing {zip_path.name} ...")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    os.system(f"unzip -q {zip_path} -d {extract_dir}")

    zarr_folders = [p for p in extract_dir.iterdir() if p.is_dir() and p.suffix == ".zarr"]
    zarr_folder = zarr_folders[0] if zarr_folders else extract_dir

    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]
    num_samples = arr.shape[0]

    with torch.inference_mode():
        sample_input = torch.zeros((1, *arr.shape[1:]), device=device)
        _, _, info = model.m.encode(sample_input)
        sample_tokens = info[2]
        if isinstance(sample_tokens, (tuple, list)):
            sample_tokens = sample_tokens[0]
        token_dim = sample_tokens.numel()

    out_zarr_folder = OUT_DIR / f"{zip_path.stem.replace('_order_images','')}_64_vector.zarr"
    if out_zarr_folder.exists():
        shutil.rmtree(out_zarr_folder)

    out_root = zarr.open(
        str(out_zarr_folder),
        mode="w",
        shape=(num_samples, token_dim),
        chunks=(batch_size * 4, token_dim),
        dtype=np.float32,
        compressor=zarr.LZ4()
    )

    dataset = ZarrDataset(arr)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=min(8, os.cpu_count()),
        pin_memory=True,
        persistent_workers=True
    )

    total_batches = len(loader)
    batches_per_save = math.ceil(total_batches / num_saves)

    start_idx = 0
    buffer = []
    buffer_count = 0

    with torch.inference_mode():
        for i, x in enumerate(tqdm(loader, desc=zip_path.name)):
            x = x.to(device, non_blocking=True)

            _, _, info = model.m.encode(x)

            tokens = info[2]
            if isinstance(tokens, (tuple, list)):
                tokens = tokens[0]

            tokens = tokens.view(x.size(0), -1).cpu().numpy()

            buffer.append(tokens)
            buffer_count += 1

            if buffer_count >= batches_per_save or i == total_batches - 1:
                chunk = np.concatenate(buffer, axis=0)

                out_root[start_idx:start_idx + chunk.shape[0], :] = chunk
                start_idx += chunk.shape[0]

                buffer = []
                buffer_count = 0

    shutil.make_archive(str(out_zarr_folder), 'zip', root_dir=out_zarr_folder)
    shutil.rmtree(extract_dir)

    #   
# -------------------------
# Main
# -------------------------
def main():
    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))
    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

if __name__ == "__main__":
    main()


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Found 270 zip files to process.
Processing GOOG_2025-11-04_order_images.zarr.zip


GOOG_2025-11-04_order_images.zarr.zip:  52%|█████▏    | 3579/6828 [18:18<16:37,  3.26it/s]


KeyboardInterrupt: 

In [15]:
import os
import math
import shutil
import zipfile
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
import zarr
import numpy as np
from tqdm import tqdm

class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)

from numcodecs import Blosc
import threading

def process_zip_file(zip_path: Path, model, device, batch_size=512):
    print(f"Processing {zip_path.name}")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)

    zarr_folders = [p for p in extract_dir.iterdir() if p.suffix == ".zarr"]
    zarr_folder = zarr_folders[0] if zarr_folders else extract_dir

    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]

    num_samples = arr.shape[0]

    # --- token size once ---
    with torch.inference_mode():
        sample = torch.zeros((1, *arr.shape[1:]), device=device)
        _, _, info = model.m.encode(sample)
        tokens = info[2]
        if isinstance(tokens, (tuple, list)):
            tokens = tokens[0]
        token_dim = tokens.numel()

    out_zarr = OUT_DIR / f"{zip_path.stem}_64_vector.zarr"

    if out_zarr.exists():
        shutil.rmtree(out_zarr)

    out = zarr.open(
        str(out_zarr),
        mode="w",
        shape=(num_samples, token_dim),
        chunks=(batch_size, token_dim),
        dtype=np.float32,
        compressor=Blosc(cname="lz4", clevel=3, shuffle=Blosc.SHUFFLE),
    )

    torch.backends.cudnn.benchmark = True

    stream = torch.cuda.Stream()

    write_idx = 0
    pending_write = None

    def async_write(start, data):
        out[start:start + data.shape[0]] = data

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):

        for i in tqdm(range(0, num_samples, batch_size), desc=zip_path.name):

            batch_np = arr[i:i + batch_size]

            # pinned staging
            batch = torch.from_numpy(batch_np).pin_memory()

            with torch.cuda.stream(stream):
                x = batch.to(device, non_blocking=True).float().div_(255)

                _, _, info = model.m.encode(x)

                tokens = info[2]
                if isinstance(tokens, (tuple, list)):
                    tokens = tokens[0]

                tokens = tokens.view(x.size(0), -1).cpu().numpy()

            torch.cuda.current_stream().wait_stream(stream)

            # async disk write
            if pending_write is not None:
                pending_write.join()

            pending_write = threading.Thread(
                target=async_write,
                args=(write_idx, tokens),
            )
            pending_write.start()

            write_idx += tokens.shape[0]

        if pending_write:
            pending_write.join()

    shutil.make_archive(str(out_zarr), "zip", root_dir=out_zarr)
    shutil.rmtree(extract_dir)
# -------------------------
# Main
# -------------------------
def main():
    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))
    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

if __name__ == "__main__":
    main()


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Found 270 zip files to process.
Processing GOOG_2025-11-04_order_images.zarr.zip


GOOG_2025-11-04_order_images.zarr.zip:   3%|▎         | 410/13656 [00:36<19:50, 11.13it/s]


KeyboardInterrupt: 

# Final efficient version

In [33]:
# Standard library
import os
import sys
import math
import re
import shutil
import zipfile
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import pytorch_lightning as pl
import zarr
from zarr.storage import ZipStore
from tqdm import tqdm
from omegaconf import OmegaConf
import argparse

# PyTorch utilities
from torch.utils.data import Dataset, DataLoader
print(os.cpu_count())
# -------------------------
# Add external repos
# -------------------------
BASE = Path("/projappl/project_2012747/mars_derrick_branch/third_party")
CKPT_DIR = Path("/scratch/project_2012747/Mars_Derrick/checkpoints/checkpoint_downsample_100")
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/intermediate/")
OUT_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

sys.path.insert(0, str(BASE / "latent_diffusion"))
sys.path.insert(0, str(BASE / "taming-transformers"))

OUT_DIR.mkdir(exist_ok=True)

# -------------------------
# Lightning wrapper
# -------------------------
class VQForOrders(pl.LightningModule):
    def __init__(self, vqmodel):
        super().__init__()
        self.m = vqmodel

    def forward(self, x):
        q, _, _ = self.m.encode(x)
        return self.m.decode(q)

# -------------------------
# Load best checkpoint
# -------------------------
def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        return float(m.group(1))
    return min(ckpts, key=extract_val_loss)

def load_model(best_ckpt):
    from ldm.util import instantiate_from_config
    cfg = OmegaConf.load(BASE / "latent_diffusion/models/first_stage_models/vq-f4/config.yaml")
    vq = instantiate_from_config(cfg.model)
    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt,
        vqmodel=vq,
        strict=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, device

# -------------------------


class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)

from numcodecs import Blosc
import threading

def process_zip_file(zip_path: Path, model, device, batch_size=1024):
    print(f"Processing {zip_path.name}")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    # Fast unzip
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)

    zarr_folders = [p for p in extract_dir.iterdir() if p.suffix == ".zarr"]
    zarr_folder = zarr_folders[0] if zarr_folders else extract_dir

    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]

    num_samples = arr.shape[0]

    # -------------------------
    # Determine token dimension once
    # -------------------------
    encode = model.m.encode

    with torch.inference_mode():
        sample = torch.zeros((1, *arr.shape[1:]), device=device)
        _, _, info = encode(sample)

        tokens = info[2]
        if isinstance(tokens, (tuple, list)):
            tokens = tokens[0]

        token_dim = tokens.numel()
        
    stem = zip_path.name
    stem = stem.replace("_order_images.zarr.zip", "")
    stem = stem.replace("_order_images.zarr", "")
    stem = stem.replace(".zarr.zip", "")   # remove leftover .zarr.zip
    stem = stem.replace(".zarr", "")       # remove leftover .zarr
    out_zarr = OUT_DIR / f"{stem}_64vectors.zarr"

    if out_zarr.exists():
        shutil.rmtree(out_zarr)

    out = zarr.open(
        str(out_zarr),
        mode="w",
        shape=(num_samples, token_dim),
        chunks=(batch_size, token_dim),
        dtype=np.float32,
        compressor=Blosc(
            cname="lz4",
            clevel=1,
            shuffle=Blosc.BITSHUFFLE
        ),
    )

    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

    stream = torch.cuda.Stream()

    write_idx = 0
    pending_write = None

    def async_write(start, data):
        out[start:start + data.shape[0]] = data

    # -------------------------
    # Double buffer setup
    # -------------------------
    next_batch_np = arr[0:batch_size]

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):

        for i in tqdm(
            range(0, num_samples, batch_size),
            desc=zip_path.name,
            smoothing=0,
            mininterval=1,
        ):

            batch_np = next_batch_np

            # Preload next batch early (IO overlap)
            next_i = i + batch_size
            if next_i < num_samples:
                next_batch_np = arr[next_i:next_i + batch_size]

            # Pinned memory staging
            batch = torch.from_numpy(batch_np).pin_memory()

            with torch.cuda.stream(stream):
                x = batch.to(
                    device,
                    non_blocking=True,
                    dtype=torch.float16
                ).div_(255)

                _, _, info = encode(x)

                tokens = info[2]
                if isinstance(tokens, (tuple, list)):
                    tokens = tokens[0]

                tokens = tokens.view(x.size(0), -1).cpu().numpy()

            torch.cuda.current_stream().wait_stream(stream)

            # Async disk write
            if pending_write is not None:
                pending_write.join()

            pending_write = threading.Thread(
                target=async_write,
                args=(write_idx, tokens),
            )
            pending_write.start()

            write_idx += tokens.shape[0]

        if pending_write:
            pending_write.join()

    shutil.make_archive(str(out_zarr), "zip", root_dir=out_zarr)
    shutil.rmtree(extract_dir)

def parse_args():
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--symbol",
        type=str,
        default=None,
        help="Process only zarr files starting with this symbol (e.g. GOOG)"
    )
    # ignore unknown args (needed in Jupyter)
    args, _ = parser.parse_known_args()
    return args
    
def main():
    args = parse_args()
    symbol = args.symbol

    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))

    # -------------------------
    # Symbol filter
    # -------------------------
    if symbol is not None:
        zip_files = [
            p for p in zip_files
            if p.name.startswith(symbol)
        ]

    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

        
if __name__ == "__main__":
    main()


40
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Found 270 zip files to process.
Processing first batch of GOOG_2025-11-04_order_images.zarr.zip
/scratch/project_2012747/mars_data/order_batch_model/train/final/GOOG_2025-11-04_64_vectors.zarr
Saved first batch to /scratch/project_2012747/mars_data/order_batch_model/train/final/GOOG_2025-11-04_64_vectors.zarr.zip
Processing first batch of AMD_2025-11-03_order_images.zarr.zip
/scratch/project_2012747/mars_data/order_batch_model/train/final/AMD_2025-11-03_64_vectors.zarr
Saved first batch to /scratch/project_2012747/mars_data/order_batch_model/train/final/AMD_2025-11-03_64_vectors.zarr.zip
Processing first batch of TSLA_2025-11-14_order_images.zarr.zip
/scratch/project_2012747/mars_data/order_b

KeyboardInterrupt: 

In [1]:
from pathlib import Path

# -------------------------
# Configuration
# -------------------------
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/raw/")

# -------------------------
# List all .zarr.zip files
# -------------------------
zip_files = list(ZIP_DIR.glob("*.zarr.zip"))

# -------------------------
# Extract symbols (prefix before first '_')
# -------------------------
symbols = set()

for p in zip_files:
    name = p.stem  # removes ".zip"
    # remove trailing ".zarr" if exists
    if name.endswith(".zarr"):
        name = name[:-5]
    # extract part before first "_"
    if "_" in name:
        sym = name.split("_")[0]
        symbols.add(sym)

# -------------------------
# Sort and display
# -------------------------
symbols = sorted(symbols)
print(f"Found {len(symbols)} unique symbols:")
print(symbols)
for s in symbols:
    print(s)

Found 15 unique symbols:
['AAPL', 'AMD', 'AMZN', 'ASML', 'AVGO', 'COST', 'GOOG', 'GOOGL', 'INTC', 'META', 'MSFT', 'NFLX', 'NVDA', 'PLTR', 'TSLA']
AAPL
AMD
AMZN
ASML
AVGO
COST
GOOG
GOOGL
INTC
META
MSFT
NFLX
NVDA
PLTR
TSLA


In [23]:
36*15

540

In [13]:
from pathlib import Path
from collections import Counter

# -------------------------
# Configuration
# -------------------------
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/raw/")

# -------------------------
# List all .zarr.zip files
# -------------------------
zip_files = list(ZIP_DIR.glob("*.zarr.zip"))

# -------------------------
# Extract symbol prefix (before first "_")
# -------------------------
symbols = []

for p in zip_files:
    name = p.stem  # removes ".zip"
    # remove trailing ".zarr" if exists
    if name.endswith(".zarr"):
        name = name[:-5]
    # extract part before first "_"
    if "_" in name:
        sym = name.split("_")[0]
        symbols.append(sym)

# -------------------------
# Count files per symbol
# -------------------------
symbol_counts_RAW = Counter(symbols)

# -------------------------
# Display sorted results 


# -------------------------
# Display sorted results 
# -------------------------
print(f"Found {len(symbol_counts_RAW)} unique symbols:\n")
for sym, count in symbol_counts_RAW.most_common():  # sorted by count descending
    print(f"{sym}: {count} files")

Found 15 unique symbols:

GOOG: 18 files
AMD: 18 files
AMZN: 18 files
INTC: 18 files
MSFT: 18 files
AAPL: 18 files
COST: 18 files
ASML: 18 files
NFLX: 18 files
AVGO: 18 files
META: 18 files
TSLA: 16 files
PLTR: 16 files
NVDA: 12 files
GOOGL: 10 files


In [15]:
from pathlib import Path
from collections import Counter

# -------------------------
# Configuration
# -------------------------
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

# -------------------------
# List all .zarr.zip files
# -------------------------
zip_files = list(ZIP_DIR.glob("*.zarr.zip"))

# -------------------------
# Extract symbol prefix (before first "_")
# -------------------------
symbols = []

for p in zip_files:
    name = p.stem  # removes ".zip"
    # remove trailing ".zarr" if exists
    if name.endswith(".zarr"):
        name = name[:-5]
    # extract part before first "_"
    if "_" in name:
        sym = name.split("_")[0]
        symbols.append(sym)

# -------------------------
# Count files per symbol
# -------------------------
symbol_counts = Counter(symbols)

# -------------------------
# Display sorted results 


IP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

# -------------------------
# List all .zarr.zip files
# -------------------------
zip_files = list(ZIP_DIR.glob("*.zarr.zip"))

# -------------------------
# Extract symbol prefix (before first "_")
# -------------------------
symbols = []

for p in zip_files:
    name = p.stem  # removes ".zip"
    # remove trailing ".zarr" if exists
    if name.endswith(".zarr"):
        name = name[:-5]
    # extract part before first "_"
    if "_" in name:
        sym = name.split("_")[0]
        symbols.append(sym)

# -------------------------
# Count files per symbol
# -------------------------
symbol_counts = Counter(symbols)

# -------------------------
# Display sorted results 
# -------------------------
print(f"Found {len(symbol_counts)} unique symbols:\n")
for sym, count in symbol_counts.most_common():  # sorted by count descending
    print(f"{sym}: {count} files")

Found 15 unique symbols:

INTC: 18 files
NFLX: 18 files
COST: 18 files
ASML: 18 files
META: 18 files
MSFT: 18 files
AVGO: 18 files
AMD: 13 files
AMZN: 12 files
TSLA: 11 files
PLTR: 10 files
GOOG: 9 files
GOOGL: 6 files
NVDA: 5 files
AAPL: 3 files


In [24]:

all_symbols = set(symbol_counts_RAW) | set(symbol_counts)
dif = 0
to_process = []
for sym in sorted(all_symbols):
    raw_count = symbol_counts_RAW.get(sym, 0)
    new_count = symbol_counts.get(sym, 0)
    diff = abs(new_count - raw_count)
    if diff >0:
        to_process.append(sym)
    dif +=diff
    print(f"{sym}: raw={raw_count}, new={new_count}, diff={diff}")

print(f"Total files to be {dif} and symbols are {to_process}")

AAPL: raw=18, new=3, diff=15
AMD: raw=18, new=13, diff=5
AMZN: raw=18, new=12, diff=6
ASML: raw=18, new=18, diff=0
AVGO: raw=18, new=18, diff=0
COST: raw=18, new=18, diff=0
GOOG: raw=18, new=9, diff=9
GOOGL: raw=10, new=6, diff=4
INTC: raw=18, new=18, diff=0
META: raw=18, new=18, diff=0
MSFT: raw=18, new=18, diff=0
NFLX: raw=18, new=18, diff=0
NVDA: raw=12, new=5, diff=7
PLTR: raw=16, new=10, diff=6
TSLA: raw=16, new=11, diff=5
Total files to be 57 and symbols are ['AAPL', 'AMD', 'AMZN', 'GOOG', 'GOOGL', 'NVDA', 'PLTR', 'TSLA']


In [32]:
def process_zip_file_first_batch(zip_path: Path, model, device, batch_size=1024):
    print(f"Processing first batch of {zip_path.name}")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    # Fast unzip
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)

    zarr_folders = [p for p in extract_dir.iterdir() if p.suffix == ".zarr"]
    zarr_folder = zarr_folders[0] if zarr_folders else extract_dir

    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]

    num_samples = arr.shape[0]
    first_batch_size = min(batch_size, num_samples)  # just the first batch

    # -------------------------
    # Determine token dimension once
    # -------------------------
    encode = model.m.encode
    with torch.inference_mode():
        sample = torch.zeros((1, *arr.shape[1:]), device=device)
        _, _, info = encode(sample)
        tokens = info[2]
        if isinstance(tokens, (tuple, list)):
            tokens = tokens[0]
        token_dim = tokens.numel()

    # -------------------------
    # Clean output file name
    # -------------------------
    # remove _order_images if present

    stem = zip_path.name
    stem = stem.replace("_order_images.zarr.zip", "")
    stem = stem.replace("_order_images.zarr", "")
    out_zarr = OUT_DIR / f"{stem}_64vectors.zarr"
    
    print(out_zarr)

    if out_zarr.exists():
        shutil.rmtree(out_zarr)

    # Force Zarr v2 to avoid v3 Blosc issues
    out = zarr.open(
        str(out_zarr),
        mode="w",
        shape=(first_batch_size, token_dim),
        chunks=(first_batch_size, token_dim),
        dtype=np.float32,
        compressor=Blosc(cname="lz4", clevel=1, shuffle=Blosc.BITSHUFFLE),
        zarr_format=2,  # important!
    )

    # -------------------------
    # Process only first batch
    # -------------------------
    batch_np = arr[0:first_batch_size]
    batch = torch.from_numpy(batch_np).to(device).float().div_(255.0)

    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        _, _, info = encode(batch)
        tokens = info[2]
        if isinstance(tokens, (tuple, list)):
            tokens = tokens[0]
        tokens = tokens.view(batch.size(0), -1).cpu().numpy()

    out[0:first_batch_size, :] = tokens

    # Save to zip and clean up
    shutil.make_archive(str(out_zarr), "zip", root_dir=out_zarr)
    shutil.rmtree(extract_dir)

    print(f"Saved first batch to {out_zarr}.zip")